<a href="https://colab.research.google.com/github/Nazirul040/Next_word_predictor_using_lstm/blob/main/Next_word_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Next word Predictor

**dataset download**

In [ ]:
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d muhammadbilalhaneef/sherlock-holmes-next-word-prediction-corpus
import zipfile
zip_ref = zipfile.ZipFile('/content/sherlock-holmes-next-word-prediction-corpus.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/muhammadbilalhaneef/sherlock-holmes-next-word-prediction-corpus
License(s): CC0-1.0
100% 222k/222k [00:00<00:00, 103MB/s]



**imports**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re

In [71]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Plan of Action --

* preprocess text
* splitting each line
*

**data handeling**

In [ ]:
with open('sherlock_holmes.txt', 'r') as f:
    text = f.read()

In [ ]:
text[:200]

'          CHAPTER I\n\n\n\n     To Sherlock Holmes she is always the woman. I have seldom heard him\n     mention her under any other name. In his eyes she eclipses and\n     predominates the whole of her s'

**text preprocessing**

In [ ]:
text = text.lower()
text = re.sub(r'\s+', ' ', text)
text = text.strip()

In [ ]:
sentences = text.split('.')

In [ ]:
sentences[:10]

['chapter i to sherlock holmes she is always the woman',
 ' i have seldom heard him mention her under any other name',
 ' in his eyes she eclipses and predominates the whole of her sex',
 ' it was not that he felt any emotion akin to love for irene adler',
 ' all emotions, and that one particularly, were abhorrent to his cold, precise but admirably balanced mind',
 ' he was, i take it, the most perfect reasoning and observing machine that the world has seen, but as a lover he would have placed himself in a false position',
 ' he never spoke of the softer passions, save with a gibe and a sneer',
 " they were admirable things for the observer--excellent for drawing the veil from men's motives and actions",
 ' but for the trained reasoner to admit such intrusions into his own delicate and finely adjusted temperament was to introduce a distracting factor which might throw a doubt upon all his mental results',
 ' grit in a sensitive instrument, or a crack in one of his own high-power lenses

In [53]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

In [54]:
vocab_size = len(tokenizer.index_word) +1
vocab_size

8178

In [55]:
max_length = max([len(sentence.split()) for sentence in sentences])
max_length

102

In [56]:
sentences = tokenizer.texts_to_sequences(sentences)
sentences[:3]

[[2067, 3, 4, 132, 34, 37, 14, 215, 1, 210],
 [3, 16, 1558, 115, 35, 3182, 36, 275, 102, 94, 204],
 [7, 13, 143, 37, 4491, 2, 4492, 1, 263, 5, 36, 4493]]

In [57]:
sequences = []
for sentence in sentences:
  if len(sentence) > 5:
    for i in range(1, len(sentence)):
      sequence = sentence[:i+1]
      sequences.append(sequence)

In [62]:
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='pre')
padded_sequences[:15]

array([[   0,    0,    0, ...,    0, 2067,    3],
       [   0,    0,    0, ..., 2067,    3,    4],
       [   0,    0,    0, ...,    3,    4,  132],
       ...,
       [   0,    0,    0, ..., 1558,  115,   35],
       [   0,    0,    0, ...,  115,   35, 3182],
       [   0,    0,    0, ...,   35, 3182,   36]], dtype=int32)

**dataset and dataloader**

In [63]:
padded_sequences = torch.tensor(padded_sequences, dtype=torch.long)
padded_sequences.shape

torch.Size([97533, 102])

In [64]:
#Splitiing data into X and y
X, y = padded_sequences[:, :-1], padded_sequences[:, -1]

In [65]:
print(X.shape)
print(y.shape)

torch.Size([97533, 101])
torch.Size([97533])


In [67]:
class CustomDataset(Dataset):
  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self):
    return len(X)

  def __getitem__(self, index):
    return self.X[index], self.y[index]

In [68]:
dataset = CustomDataset(X, y)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

**Model structure**

In [69]:
vocab_size

8178

In [109]:
class Model(nn.Module):
  def __init__(self, vocab_size, embedding_dim=128):
    super(Model, self).__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.lstm = nn.LSTM(embedding_dim, 512, batch_first=True)
    self.fc = nn.Linear(512, vocab_size)

  def forward(self, x):
    # x shape: [batch_size, sequence_length]
    embedded = self.embedding(x)
    # embedded shape: [batch_size, sequence_length, embedding_dim]
    lstm_out, (h_n, c_n) = self.lstm(embedded)
    # lstm_out shape: [batch_size, sequence_length, hidden_dim]
    last_timestep_out = lstm_out[:, -1, :]
    # last_timestep_out shape: [batch_size, hidden_dim]
    logits = self.fc(last_timestep_out)
    # logits shape: [batch_size, vocab_size]

    return logits

That is a brilliant question. If h_n is literally the exact same final thought as lstm_out[:, -1, :], why slice a massive tensor when PyTorch hands you the final state on a silver platter?

You absolutely can use h_n, and many developers do! However, there is a very specific PyTorch quirk regarding the batch_first=True argument that makes using lstm_out slightly cleaner for this specific architecture.

Here is the breakdown of why I chose lstm_out over h_n for your model.

The batch_first Quirk
When you initialized your LSTM, you explicitly told it batch_first=True.

This tells PyTorch: "Hey, my input sequences and output sequences should always have the batch_size as the very first dimension, because that's how my DataLoader gives it to me."

The Catch: PyTorch applies this rule to lstm_out, but it ignores this rule for h_n and c_n.

The Shape Mismatch
Because of that quirk, the tensors come out looking like this:

Using lstm_out:

lstm_out shape: [batch_size, sequence_length, hidden_dim]

Slice it: lstm_out[:, -1, :]

Resulting shape: [batch_size, hidden_dim] * This perfectly matches what your Linear layer wants!

Using h_n:

h_n shape: [num_layers, batch_size, hidden_dim]

Even though we only have 1 layer, PyTorch keeps that dimension there. So the shape is actually [1, 64, 256].

If you try to pass h_n directly into your Linear layer, it will crash because it has 3 dimensions instead of 2.

To fix it, you have to manually strip away that first dimension by writing h_n.squeeze(0) or h_n[-1, :, :].

Which one should you use?
Use lstm_out[:, -1, :] when you are doing standard Next-Word Prediction or Sentiment Analysis. It keeps your code clean, avoids dimension-squeezing gymnastics, and guarantees your batch size is always the first dimension.

Use h_n when you are building complex Sequence-to-Sequence models (like English-to-Spanish translation). In those architectures, you have two LSTMs (an Encoder and a Decoder). The Decoder explicitly expects its incoming memory to have that [num_layers, batch_size, hidden_dim] shape, so passing h_n directly is required.

Since we are just doing simple next-word prediction, slicing lstm_out is the path of least resistance!

In [110]:
model = Model(vocab_size=vocab_size).to(device)
print(model)

Model(
  (embedding): Embedding(8178, 128)
  (lstm): LSTM(128, 512, batch_first=True)
  (fc): Linear(in_features=512, out_features=8178, bias=True)
)


**setting up hyperparameters**

In [111]:
epochs = 200
learning_rate = 0.001

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

**training loop**

In [112]:
for epoch in range(epochs):
  total_loss = 0
  for X, y in dataloader:
    X = X.to(device)
    y = y.to(device)

    optimizer.zero_grad()

    y_pred = model(X)

    batch_loss = loss_fn(y_pred, y)

    batch_loss.backward()
    optimizer.step()

    total_loss += batch_loss.item()

  print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader)}")

Epoch 1/200, Loss: 9.020837783813477
Epoch 2/200, Loss: 8.903986930847168
Epoch 3/200, Loss: 8.7702054977417
Epoch 4/200, Loss: 8.568338394165039
Epoch 5/200, Loss: 8.176990509033203
Epoch 6/200, Loss: 7.476391315460205
Epoch 7/200, Loss: 6.679703712463379
Epoch 8/200, Loss: 5.890617370605469
Epoch 9/200, Loss: 5.246683597564697
Epoch 10/200, Loss: 4.834053039550781
Epoch 11/200, Loss: 4.56726598739624
Epoch 12/200, Loss: 4.387481689453125
Epoch 13/200, Loss: 4.268058776855469
Epoch 14/200, Loss: 4.177742958068848
Epoch 15/200, Loss: 4.108644485473633
Epoch 16/200, Loss: 4.048551082611084
Epoch 17/200, Loss: 4.0058746337890625
Epoch 18/200, Loss: 3.973121404647827
Epoch 19/200, Loss: 3.940498113632202
Epoch 20/200, Loss: 3.8915209770202637
Epoch 21/200, Loss: 3.8470592498779297
Epoch 22/200, Loss: 3.800703525543213
Epoch 23/200, Loss: 3.7542219161987305
Epoch 24/200, Loss: 3.7065773010253906
Epoch 25/200, Loss: 3.660130262374878
Epoch 26/200, Loss: 3.6111884117126465
Epoch 27/200, Loss

In [113]:
'''
['chapter i to sherlock holmes she is always the woman',
 ' i have seldom heard him mention her under any other name',
 ' in his eyes she eclipses and predominates the whole of her sex',
 ' it was not that he felt any emotion akin to love for irene adler',
 ' all emotions, and that one particularly, were abhorrent to his cold, precise but admirably balanced mind',
 ' he was, i take it, the most perfect reasoning and observing machine that the world has seen, but as a lover he would have placed himself in a false position',
 ' he never spoke of the softer passions, save with a gibe and a sneer',
 " they were admirable things for the observer--excellent for drawing the veil from men's motives and actions",
 ' but for the trained reasoner to admit such intrusions into his own delicate and finely adjusted temperament was to introduce a distracting factor which might throw a doubt upon all his mental results',
 ' grit in a sensitive instrument, or a crack in one of his own high-power lenses, would not be more disturbing than a strong emotion in a nature such as his']
 '''

'\n[\'chapter i to sherlock holmes she is always the woman\',\n \' i have seldom heard him mention her under any other name\',\n \' in his eyes she eclipses and predominates the whole of her sex\',\n \' it was not that he felt any emotion akin to love for irene adler\',\n \' all emotions, and that one particularly, were abhorrent to his cold, precise but admirably balanced mind\',\n \' he was, i take it, the most perfect reasoning and observing machine that the world has seen, but as a lover he would have placed himself in a false position\',\n \' he never spoke of the softer passions, save with a gibe and a sneer\',\n " they were admirable things for the observer--excellent for drawing the veil from men\'s motives and actions",\n \' but for the trained reasoner to admit such intrusions into his own delicate and finely adjusted temperament was to introduce a distracting factor which might throw a doubt upon all his mental results\',\n \' grit in a sensitive instrument, or a crack in on

In [114]:
def prediction(text):
  tokenized_text = tokenizer.texts_to_sequences([text])
  padded_text = pad_sequences(tokenized_text, maxlen=max_length, padding='pre')

  y_pred = model(torch.tensor(padded_text, dtype=torch.long).to(device))
  predicted_index = torch.argmax(y_pred, dim=1).item()
  predicted_word = tokenizer.index_word[predicted_index]

  return predicted_word

In [118]:
num_tokens = 10
input_text = "and that one particularly"
print(input_text, end=' ')
for i in range(num_tokens):
  output = prediction(input_text)
  print(output, end=' ')
  input_text += " " + output


and that one particularly were abhorrent to his cold precise but admirably balanced mind 